# Web scraping des matchs de football des Coupes du Monde (1930-2026)

Notebook de l'article [Web Scraping des matchs de football des Coupes du Monde 1930 à 2026 avec Python](https://moncoachdata.com/blog/web-scraping-des-matchs-de-football/) (MonCoachData).

On récupère depuis Wikipédia tous les matchs de toutes les Coupes du Monde avec `requests` et Beautiful Soup, puis on les nettoie et on les analyse avec Pandas. Exécute les cellules **dans l'ordre** : il faut une connexion internet, et l'exécution complète prend environ 2 à 3 minutes (une pause d'une demi-seconde entre deux pages, pour ne pas surcharger Wikipédia).

Si tu travailles sur ton ordinateur, installe d'abord les bibliothèques (déjà présentes dans Google Colab) : `pip install beautifulsoup4 lxml requests pandas`.

## Télécharger la page de la Coupe du Monde 2014

On s'identifie avec un en-tête `User-Agent`, comme le demande Wikipédia. Le code `200` signifie que la page a bien été téléchargée.

In [1]:
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "MonCoachData-tutoriel/1.0 (https://moncoachdata.com)"}
URL = "https://en.wikipedia.org/wiki/2014_FIFA_World_Cup"

response = requests.get(URL, headers=HEADERS, timeout=30)
response.status_code

200

## Créer la soupe et trouver les matchs

Chaque match est une boîte `div` de classe `footballbox`. Attendu : **16** matchs (la phase finale de 2014).

In [2]:
soup = BeautifulSoup(response.text, "lxml")
soup.title.get_text()

'2014 FIFA World Cup - Wikipedia'

In [3]:
matches = soup.find_all("div", class_="footballbox")
len(matches)

16

## Lire les trois cellules d'un match

Les noms d'équipes contiennent une espace insécable (`\xa0`) : on la supprime avec `.strip()`.

In [4]:
premier = matches[0]

print(repr(premier.find("th", class_="fhome").get_text()))
print(repr(premier.find("th", class_="fscore").get_text()))
print(repr(premier.find("th", class_="faway").get_text()))

'Brazil\xa0'
'1–1 (a.e.t.)'
'\xa0Chile'


In [5]:
def texte(element):
    return element.get_text().strip()


home, score, away = [], [], []

for match in matches:
    home.append(texte(match.find("th", class_="fhome")))
    score.append(texte(match.find("th", class_="fscore")))
    away.append(texte(match.find("th", class_="faway")))

print(home[0], "|", score[0], "|", away[0])

Brazil | 1–1 (a.e.t.) | Chile


## Créer le DataFrame de 2014

In [6]:
import pandas as pd

df_2014 = pd.DataFrame({"home": home, "score": score, "away": away})
df_2014["year"] = 2014
df_2014.head(8)

,home,score,away,year
0,Brazil,1–1 (a.e.t.),Chile,2014
1,Colombia,2–0,Uruguay,2014
2,Netherlands,2–1,Mexico,2014
3,Costa Rica,1–1 (a.e.t.),Greece,2014
4,France,2–0,Nigeria,2014
5,Germany,2–1 (a.e.t.),Algeria,2014
6,Argentina,1–0 (a.e.t.),Switzerland,2014
7,Belgium,2–1 (a.e.t.),United States,2014


In [7]:
df_2014.to_csv("fifa_worldcup_2014.csv", index=False)

## Première version pour toutes les éditions : il manque des matchs

Attendu : seulement **16** matchs par édition à partir de 1998.

In [8]:
for year in [2014, 2018, 2022]:
    print(f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup")

https://en.wikipedia.org/wiki/2014_FIFA_World_Cup
https://en.wikipedia.org/wiki/2018_FIFA_World_Cup
https://en.wikipedia.org/wiki/2022_FIFA_World_Cup


In [9]:
def get_matches_v1(year):
    url = f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup"
    response = requests.get(url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(response.text, "lxml")
    matches = soup.find_all("div", class_="footballbox")
    return pd.DataFrame({
        "home": [texte(m.find("th", class_="fhome")) for m in matches],
        "score": [texte(m.find("th", class_="fscore")) for m in matches],
        "away": [texte(m.find("th", class_="faway")) for m in matches],
        "year": year,
    })


years_v1 = [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974,
            1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014,
            2018, 2022]
df_v1 = pd.concat([get_matches_v1(year) for year in years_v1], ignore_index=True)
df_v1.groupby("year").size().tail(9)

,0
year,
1990,52
1994,52
1998,16
2002,16
2006,16
2010,16
2014,16
2018,16
2022,16


## Les groupes sont sur des pages séparées depuis 1998

Attendu : **6** matchs pour le groupe A de 2014.

In [10]:
url_groupe = "https://en.wikipedia.org/wiki/2014_FIFA_World_Cup_Group_A"
soup_groupe = BeautifulSoup(requests.get(url_groupe, headers=HEADERS, timeout=30).text, "lxml")
matchs_groupe = soup_groupe.find_all("div", class_="footballbox")
len(matchs_groupe)

6

## La fonction complète get_matches()

Elle lit la page principale et, depuis 1998, les pages de groupes (8, puis 12 en 2026). Attendu pour 2014 : **(64, 4)**.

In [11]:
import time
from string import ascii_uppercase

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "MonCoachData-tutoriel/1.0 (https://moncoachdata.com)"}


def texte(element):
    return element.get_text().strip()


def get_page_matches(url, year):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    return [
        {
            "home": texte(match.find("th", class_="fhome")),
            "score": texte(match.find("th", class_="fscore")),
            "away": texte(match.find("th", class_="faway")),
            "year": year,
        }
        for match in soup.find_all("div", class_="footballbox")
    ]


def get_matches(year):
    base = f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup"
    pages = [base]
    if year >= 1998:
        nb_groupes = 12 if year >= 2026 else 8
        pages = [f"{base}_Group_{lettre}" for lettre in ascii_uppercase[:nb_groupes]] + pages
    lignes = []
    for url in pages:
        lignes += get_page_matches(url, year)
        time.sleep(0.5)
    return pd.DataFrame(lignes)

In [12]:
df_2014_complet = get_matches(2014)
df_2014_complet.shape

(64, 4)

## Données historiques (1930-2018), Coupe du Monde 2022 et Coupe du Monde 2026

Attendu : **(901, 4)**, **(64, 4)** puis **(104, 4)**. Compte 1 à 2 minutes pour la première cellule.

In [13]:
years = [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974,
         1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014,
         2018]

df_fifa = pd.concat([get_matches(year) for year in years], ignore_index=True)
df_fifa.to_csv("fifa_worldcup_historical_data.csv", index=False)
df_fifa.shape

(901, 4)

In [14]:
df_2022 = get_matches(2022)
df_2022.to_csv("fifa_worldcup_fixture.csv", index=False)
df_2022.shape

(64, 4)

In [15]:
df_2026 = get_matches(2026)
df_2026.to_csv("fifa_worldcup_2026.csv", index=False)
df_2026.shape

(104, 4)

In [16]:
df_2026.tail(2)

,home,score,away,year
102,France,4–6,England,2026
103,Spain,1–0 (a.e.t.),Argentina,2026


## Nettoyer les scores

Le match Suède-Autriche de 1938 est annulé (`Cancelled`). Attendu après nettoyage : **(1068, 8)**.

In [17]:
df_fifa[df_fifa["score"] == "Cancelled"]

,home,score,away,year
37,Sweden,Cancelled,Austria,1938


In [18]:
df = pd.concat([df_fifa, df_2022, df_2026], ignore_index=True)
df = df[df["score"] != "Cancelled"].copy()

buts = df["score"].str.extract(r"(\d+)–(\d+)").astype(int)
df["home_goals"] = buts[0]
df["away_goals"] = buts[1]
df["buts"] = df["home_goals"] + df["away_goals"]
df["prolongation"] = df["score"].str.contains("a.e.t.", regex=False)
df.shape

(1068, 8)

## Analyser : statistiques par édition

In [19]:
stats = df.groupby("year").agg(
    matchs=("buts", "size"),
    buts=("buts", "sum"),
    prolongations=("prolongation", "sum"),
)
stats["buts_par_match"] = (stats["buts"] / stats["matchs"]).round(2)
stats

,matchs,buts,prolongations,buts_par_match
year,,,,
1930,18,70,0,3.89
1934,17,70,3,4.12
1938,18,84,6,4.67
1950,22,88,0,4.00
1954,26,140,3,5.38
1958,35,126,1,3.60
1962,32,89,0,2.78
1966,32,89,1,2.78
1970,32,95,3,2.97


In [20]:
stats.sort_values("buts_par_match", ascending=False).head(3)

,matchs,buts,prolongations,buts_par_match
year,,,,
1954,26,140,3,5.38
1938,18,84,6,4.67
1934,17,70,3,4.12


## Que disent les données de la Coupe du Monde 2026 ?

Classement de l'édition par buts par match, puis buts en phase de groupes et en phase finale.

In [21]:
rang = stats["buts_par_match"].rank(ascending=False).astype(int)
print("2026 :", stats.loc[2026, "buts_par_match"], "buts par match,", f"{rang[2026]}e sur {len(stats)} éditions")
print("2022 :", stats.loc[2022, "buts_par_match"], "buts par match,", f"{rang[2022]}e sur {len(stats)} éditions")

2026 : 2.96 buts par match, 8e sur 23 éditions
2022 : 2.69 buts par match, 13e sur 23 éditions


In [22]:
df_2026_clean = df[df["year"] == 2026].copy()
# get_matches() range les 72 matchs de groupes avant les 32 de la phase finale
df_2026_clean["phase"] = ["Groupes"] * 72 + ["Phase finale"] * 32
df_2026_clean.groupby("phase")["buts"].agg(matchs="count", buts="sum", moyenne="mean").round(2)

,matchs,buts,moyenne
phase,,,
Groupes,72,215,2.99
Phase finale,32,93,2.91


## Tout le code

In [23]:
import time
from string import ascii_uppercase

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "MonCoachData-tutoriel/1.0 (https://moncoachdata.com)"}


def texte(element):
    return element.get_text().strip()


def get_page_matches(url, year):
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    return [
        {
            "home": texte(match.find("th", class_="fhome")),
            "score": texte(match.find("th", class_="fscore")),
            "away": texte(match.find("th", class_="faway")),
            "year": year,
        }
        for match in soup.find_all("div", class_="footballbox")
    ]


def get_matches(year):
    base = f"https://en.wikipedia.org/wiki/{year}_FIFA_World_Cup"
    pages = [base]
    if year >= 1998:
        nb_groupes = 12 if year >= 2026 else 8
        pages = [f"{base}_Group_{lettre}" for lettre in ascii_uppercase[:nb_groupes]] + pages
    lignes = []
    for url in pages:
        lignes += get_page_matches(url, year)
        time.sleep(0.5)
    return pd.DataFrame(lignes)


# Données historiques (1930-2018) et Coupe du Monde 2022
years = [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974,
         1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014,
         2018]
df_fifa = pd.concat([get_matches(year) for year in years], ignore_index=True)
df_fifa.to_csv("fifa_worldcup_historical_data.csv", index=False)

df_2022 = get_matches(2022)
df_2022.to_csv("fifa_worldcup_fixture.csv", index=False)

# Coupe du Monde 2026
df_2026 = get_matches(2026)
df_2026.to_csv("fifa_worldcup_2026.csv", index=False)

# Nettoyage
df = pd.concat([df_fifa, df_2022, df_2026], ignore_index=True)
df = df[df["score"] != "Cancelled"].copy()

buts = df["score"].str.extract(r"(\d+)–(\d+)").astype(int)
df["home_goals"] = buts[0]
df["away_goals"] = buts[1]
df["buts"] = df["home_goals"] + df["away_goals"]
df["prolongation"] = df["score"].str.contains("a.e.t.", regex=False)

# Analyse
stats = df.groupby("year").agg(
    matchs=("buts", "size"),
    buts=("buts", "sum"),
    prolongations=("prolongation", "sum"),
)
stats["buts_par_match"] = (stats["buts"] / stats["matchs"]).round(2)

print(df.shape)
stats.tail(3)

(1068, 8)


,matchs,buts,prolongations,buts_par_match
year,,,,
2018,64,169,5,2.64
2022,64,172,5,2.69
2026,104,308,9,2.96
